# 02 - Baseline CNN Classification

This notebook trains a simple CNN from scratch at 128x128 resolution and evaluates it on the official test split.

## Google Colab Setup

Run the next cell only when using Google Colab. It mounts Google Drive, moves into the project folder, and installs the extra packages Colab may not already include. If you run locally, skip it.


In [ ]:
# Colab-only setup. Skip this cell when running locally.
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/PetVision-DeepLearning')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'tensorflow-datasets', 'seaborn', 'scikit-learn', 'streamlit', 'opencv-python'
    ])
    print('Colab project root:', os.getcwd())
else:
    print('Not running in Google Colab. Continue with the local setup cells below.')


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import get_splits, configure_for_performance, save_label_mapping
from src.preprocessing import preprocess_baseline_classification
from src.models_classification import build_baseline_cnn, compile_classifier
from src.training import standard_callbacks
from src.evaluation import collect_predictions, save_classification_outputs, top_k_accuracy
from src.visualization import plot_training_history, plot_confusion_matrix

In [ ]:
BATCH_SIZE = 32
EPOCHS = 15

train_raw, val_raw, test_raw, info, label_names = get_splits()
save_label_mapping(label_names, PROJECT_ROOT / "results/figures/label_mapping.json")

train_ds = configure_for_performance(train_raw.map(preprocess_baseline_classification, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE, shuffle=True)
val_ds = configure_for_performance(val_raw.map(preprocess_baseline_classification, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)
test_ds = configure_for_performance(test_raw.map(preprocess_baseline_classification, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)

In [ ]:
model = build_baseline_cnn(input_shape=(128, 128, 3), num_classes=len(label_names))
compile_classifier(model, learning_rate=1e-3)
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/baseline_classifier.keras", patience=4),
)
plot_training_history(history, PROJECT_ROOT / "results/classification/baseline_training_curves.png", "Baseline CNN")

In [ ]:
test_metrics = model.evaluate(test_ds, verbose=1)
print(dict(zip(model.metrics_names, test_metrics)))

y_true, y_pred, y_prob = collect_predictions(model, test_ds)
print("Top-3 accuracy:", top_k_accuracy(y_true, y_prob, k=3))
save_classification_outputs(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/baseline")
plot_confusion_matrix(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/baseline_confusion_matrix.png")

In [ ]:
# Show wrong predictions.
wrong_indices = np.where(y_true != y_pred)[0][:12]
images, labels = [], []
for batch_images, batch_labels in test_ds.unbatch().take(500):
    images.append(batch_images.numpy())
    labels.append(int(batch_labels.numpy()))
images = np.array(images)
labels = np.array(labels)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, idx in zip(axes.ravel(), wrong_indices):
    ax.imshow(images[idx])
    ax.set_title(f"True: {label_names[y_true[idx]]}\nPred: {label_names[y_pred[idx]]}", fontsize=8)
    ax.axis("off")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/classification/baseline_wrong_predictions.png", dpi=160)
plt.show()